# Aula 4 — Alocar tarefas e formar uma dupla

Distribuir tarefas por capacidade e fazer uma conferência que exige duas pessoas.

Pré-requisito: funções, listas e dicionários usados no M3. Execute na ordem; depois mude uma condição do exercício.


## Preparar o Colab

Execute a instalação abaixo e espere terminar. `%pip` instala a biblioteca no Python deste notebook; `==` fixa a versão usada na aula. Se o Colab pedir reinício, reinicie e execute desde o início.

Esta atividade usa regras em Python. Ela não precisa de chave e não chama uma IA. `RunnableLambda` permite colocar uma função comum em um fluxo do LangChain; a função continua sendo executada por Python.


In [1]:
%pip install -q langchain-core==1.7.1


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Dados e alcance

A referência preparada traz T1 aceita por Ana. A conferência T4 depende de T1 e exige estoque+logística em duas pessoas. Nesta fase Bruno ainda cuida de T2; Ana e Carla estão disponíveis para a conferência. Não há cronograma ou cálculo de rota neste núcleo.


## 1. Preparar a alocação

Vamos expor a escolha de pessoas e a regra da dupla. O resultado é um plano fictício, sem executar expedição.


In [2]:
# RunnableLambda transforma uma função Python comum em uma etapa executável dentro do fluxo do LangChain.
from langchain_core.runnables import RunnableLambda

# combinations gera duplas possíveis a partir de uma lista, sem repetir a mesma dupla em ordens diferentes.
from itertools import combinations


## 2. Ler a referência de entrada

T1 confere camisetas, T2 planeja coleta e T3 separa embalagem. A carga conta tarefas atribuídas, sem representar horas ou duração.


In [3]:
# Define as capacidades de cada pessoa usando conjuntos (sets) para evitar duplicatas.
capacidades = {'Ana':
                    {'estoque'},
               'Bruno':
                    {'logistica'},
               'Carla':
                    {'estoque', 'logistica'}}

# Lista de tarefas onde cada tupla contém o identificador da tarefa e a habilidade exigida.
tarefas = [('T1', 'estoque'), ('T2', 'logistica'), ('T3', 'estoque')]

# Simula um compromisso de negociação já fechado para a realização da tarefa T1.
compromisso_preparado = {'pedido_id': 'P001', 'responsavel': 'Ana', 'estado': 'alocada'}

# Estrutura inicial que servirá de entrada para o fluxo, contendo as pessoas ativas, as pessoas livres para a conferência e o compromisso.
entrada = {'compromisso_t1': compromisso_preparado, 'ativos': ['Ana', 'Bruno', 'Carla'],
           'livres_conferencia': ['Ana', 'Carla']}

# Imprime os dados preparados para verificação.
print('Referência preparada:', entrada)


Referência preparada: {'compromisso_t1': {'pedido_id': 'P001', 'responsavel': 'Ana', 'estado': 'alocada'}, 'ativos': ['Ana', 'Bruno', 'Carla'], 'livres_conferencia': ['Ana', 'Carla']}


## 3. Formar uma dupla capaz

`combinations(..., 2)` visita duplas. `set` representa capacidades sem repetição. `update` reúne as capacidades das duas pessoas. A dupla precisa cobrir estoque e logística.


In [4]:
def formar_coalizao(participantes):
    # Itera sobre todas as combinações de 2 pessoas da lista de participantes ordenada.
    for dupla in combinations(sorted(participantes), 2):
        primeira = dupla[0] # Primeira pessoa da dupla candidata
        segunda = dupla[1]  # Segunda pessoa da dupla candidata

        # Inicia o conjunto de habilidades da dupla usando as habilidades da primeira pessoa
        cobertura = set(capacidades[primeira])
        # Adiciona as habilidades da segunda pessoa ao conjunto (unindo as capacidades)
        cobertura.update(capacidades[segunda])

        # Verifica se, juntas, essas duas pessoas cobrem tanto "estoque" quanto "logistica"
        if "estoque" in cobertura and "logistica" in cobertura:
            return [primeira, segunda] # Retorna a dupla assim que encontra a primeira válida

    return [] # Retorna lista vazia se nenhuma dupla conseguir suprir os requisitos

# Testa a função com a lista de pessoas livres e imprime o resultado
print("Dupla candidata:", formar_coalizao(entrada["livres_conferencia"]))


Dupla candidata: ['Ana', 'Carla']


## 4. Escolher sem fabricar um aceite

T1 precisa do compromisso recebido. Nas outras tarefas, a regra escolhe menor carga e desempata pelo nome.


In [5]:
def alocar(entrada):
    carga = {} # Dicionário para rastrear quantas tarefas cada pessoa recebeu (balanceamento)
    plano = [] # Lista que vai armazenar o plano de trabalho final

    # Inicia a contagem de tarefas de todos os ativos como zero
    for nome in entrada["ativos"]:
        carga[nome] = 0

    preferida = None # Variável para guardar a pessoa já comprometida para T1
    compromisso = entrada["compromisso_t1"] # Puxa a informação de compromisso de T1

    # Se há compromisso fechado (alocada), define o responsável como a pessoa preferida
    if compromisso is not None:
        if compromisso.get("estado") == "alocada":
            preferida = compromisso["responsavel"]

    # Itera sobre as tarefas para tentar alocá-las uma a uma
    for tarefa_id, requisito in tarefas:
        aptos = [] # Lista das pessoas que possuem a habilidade exigida
        for nome in sorted(carga):
            if requisito in capacidades[nome]:
                aptos.append(nome)

        escolhida = None # Guarda quem será o escolhido final para esta tarefa

        if tarefa_id == "T1":
            # Se for a T1, verificamos se a pessoa 'preferida' está na lista de pessoas aptas
            if preferida in aptos:
                escolhida = preferida
        else:
            # Para as outras tarefas, escolhe-se a pessoa apta com menor carga de tarefas atual
            for nome in aptos:
                if escolhida is None or carga[nome] < carga[escolhida]:
                    escolhida = nome

        equipe = [] # Equipe designada para a tarefa (normalmente apenas a pessoa escolhida)
        estado = "sem_capacidade" # Assume erro por falta de capacidade inicialmente

        # Falha específica: se for a T1 e ninguém for escolhido, marca como sem acordo
        if tarefa_id == "T1":
            estado = "sem_acordo"

        # Se conseguimos escolher alguém, adiciona à equipe, aumenta sua carga e muda o estado
        if escolhida is not None:
            equipe.append(escolhida)
            carga[escolhida] += 1
            estado = "alocada"

        # Adiciona o resultado da alocação desta tarefa ao plano geral
        plano.append({"tarefa": tarefa_id, "equipe": equipe, "estado": estado})

    # Clona a entrada para não modificá-la diretamente, adiciona o ID e o plano e a devolve
    resposta = entrada.copy()
    resposta["pedido_id"] = "P001"
    resposta["plano"] = plano
    return resposta


## 5. Conferir T1 antes de usar a dupla

Esta segunda função completa o plano. T4 depende de T1 e das pessoas livres nesta fase.


In [6]:
def concluir_conferencia(dados):
    plano = dados["plano"].copy() # Copia o plano existente das tarefas (T1, T2, T3)

    # Verifica se a T1 foi concluída (se a equipe da T1 possui pessoas)
    # Aqui assume-se que T1 é o primeiro elemento (índice 0) do plano
    t1_pronta = len(plano[0]["equipe"]) > 0

    livres = [] # Lista das pessoas que estão ativas e livres para fazer a conferência (T4)
    for nome in dados["livres_conferencia"]:
        if nome in dados["ativos"]:
            livres.append(nome)

    dupla = []
    estado = "dependencia_pendente" # Se T1 não foi feita, a T4 fica aguardando (pendente)

    # Só tenta montar a equipe da T4 se a T1 estiver pronta
    if t1_pronta:
        dupla = formar_coalizao(livres) # Invoca a função que acha 2 pessoas com as habilidades
        estado = "sem_capacidade" # Assume que não há capacidade até achar a dupla
        if dupla:
            estado = "alocada" # Tendo a dupla, muda para alocada com sucesso

    # Insere a tarefa T4 no plano, informando de quem é a equipe e apontando a dependência da T1
    plano.append({"tarefa": "T4", "equipe": dupla, "estado": estado, "depende": ["T1"]})

    # Retorna o dicionário de dados atualizado com o plano final completo
    resposta = dados.copy()
    resposta["plano"] = plano
    return resposta


## 6. Ver o plano inteiro

A dependência preserva o compromisso da negociação. Uma dupla disponível não pode resolver sozinha a ausência de aceite de T1.


In [7]:
# Une as duas funções (alocar e concluir_conferencia) em um único fluxo de execução encadeado (pipeline)
fluxo_alocacao = RunnableLambda(alocar) | RunnableLambda(concluir_conferencia)

# Invoca todo o fluxo passando os dados de entrada que configuramos no passo 2
resultado = fluxo_alocacao.invoke(entrada)

# Imprime passo a passo cada tarefa e quem foi designado, mostrando o resultado do plano final
for tarefa in resultado['plano']:
    print(tarefa)


{'tarefa': 'T1', 'equipe': ['Ana'], 'estado': 'alocada'}
{'tarefa': 'T2', 'equipe': ['Bruno'], 'estado': 'alocada'}
{'tarefa': 'T3', 'equipe': ['Carla'], 'estado': 'alocada'}
{'tarefa': 'T4', 'equipe': ['Ana', 'Carla'], 'estado': 'alocada', 'depende': ['T1']}


## Sua pequena alteração

Troque `compromisso_t1` por `None` e execute desde a entrada. T1 e T4 devem ficar pendentes. T2 e T3 ainda podem receber pessoas?

Anote o que previu e compare com a saída depois de executar. Para refazer o caso-base, execute desde os dados de entrada.
